## actualizar retiro telef 

In [75]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [45]:
query = f"""
    select * from Alice.prospectos_correos_alfin
    where fecha_envio>='2026-08-01'
"""
df_prospectos_envio = pd.read_sql(query, engine_mysql)
df_prospectos_envio.shape

(44187, 21)

In [49]:
filename='BLOQUEO AGENDA.xlsx'
filePath = os.path.join(ruta_alfin, filename)
df_hoy = pd.read_excel(filePath)

In [50]:
df_hoy["dni_cliente"] = (
    df_hoy["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)

In [51]:
df_hoy=df_hoy.drop_duplicates(subset=['dni_cliente'])

In [54]:
print(df_hoy.shape)
print(df_hoy.merge(df_prospectos_envio,on='dni_cliente',how='inner').drop_duplicates(subset=['dni_cliente']).shape)

(70, 2)
(68, 22)


In [55]:
df_seg=df_prospectos_envio.merge(df_hoy[['dni_cliente']],on='dni_cliente',how='inner').drop_duplicates(subset=['dni_cliente'])

In [4]:
df_prospectos_envio = df_prospectos_envio.drop_duplicates(
    subset=["dni_cliente"]
)

In [5]:
df_prospectos_envio.shape

(21149, 21)

In [7]:
query = f"""
	select NUMERO_DOCUMENTO as dni_cliente,color_final from DANTALION.dbo.Base_Maestra_Alfin_bk_Vigente
"""
df_maestra = pd.read_sql(query, engine_kishin)

df_maestra["dni_cliente"] = (
    df_maestra["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_maestra.columns.tolist())


['dni_cliente', 'color_final']


In [ ]:
filename='AQUI.xlsx'
filePath = os.path.join(ruta_csv, filename)
df_formato = pd.read_excel(filePath)
df_formato.colums

In [ ]:
filename='fomato_agendas_alfin_credicash_2026.xlsx'
filePath = os.path.join(ruta_csv, filename)
df_formato = pd.read_excel(filePath)
df_formato['supervisor']='CARLOS ENRIQUE RAMIREZ CACHIQUE'
df_formato['canal_campo']='CALL CENTER / TARGET OUTSOURCING'
df_formato['codigo_ejecutivo_id']='00000001'
df_formato['ejecutivo_target']='BOT'
df_formato['cdv_alfin_banco']='ROSA HONOR'

df_formato = df_formato.rename(columns={
    'monto': 'monto_solicitado'
})
df_formato["dni_cliente"] = (
    df_formato["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
# df_formato = df_formato.merge(
#     df_maestra,
#     on='dni_cliente',
#     how='left'
# )

df_formato['fecha_visita']='2026-07-27'

# Semilla opcional para reproducibilidad
# np.random.seed(123)

# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_formato))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_formato))

# Crear la columna
df_formato["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

df_formato['telefono_cliente']=df_formato['celular']
df_formato['dni_vendedor']=df_formato['ejecutivo_target']
df_formato['agencia_tienda']=df_formato['cod_agencia']
df_formato['operador']='TARGET'
df_formato['tipo_gestion']='Derivacion'

In [22]:
df_formato.drop_duplicates(subset=["dni_cliente"], inplace=True)



In [23]:
df_formato.shape

(6129, 19)

In [6]:
# df_formato = df_formato[
#     df_formato['agencia_atencion'].isin([
#         'SAN JUAN DE LURIG',
#         'ENMANCIPACION',
#         'PC HUANCAYO',
#         'TRUJ CENTRO',
#         'PC TACNA',
#         'PC HUARAZ',
#         'TRUJ AMERICA',
#         'AREQ CAYMA',
#         'AREQ PAMPILLA'
#     ])
# ]

#### Validar el nombre de la agencia

In [24]:
df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace("CAÃ‘ETE", "CAÑETE")
)

In [28]:
query = f"""
	select * from Alice.agencias_alfin
"""
df_agencia = pd.read_sql(query, engine_mysql)

set_correo = set(
    df_formato['agencia_atencion']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_correo']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

set()
set()


In [26]:
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'AREQUIPA CAYMA', 'TRUJILLO CENTRO', 'TACNA', 'HUARAZ', 'SAN JUAN DE LURIGANCHO', 'TRUJILLO AMERICA', 'AREQUIPA PAMPILLA', 'HUANCAYO', 'EMANCIPACION'}
{'AREQ PAMPILLA', 'PC HUARAZ', 'ENMANCIPACION', 'AREQ CAYMA', 'TRUJ CENTRO', 'SAN JUAN DE LURIG', 'TRUJ AMERICA', 'PC HUANCAYO', 'PC TACNA'}


#### validar el codigo de agencia 

In [12]:

set_correo = set(
    df_formato['agencia_tienda']
    .dropna()
    .drop_duplicates()
)

set_agencia = set(
    df_agencia['agencia_Formulario']
    .dropna()
    .drop_duplicates()
)
# print(set_correo & set_agencia)
print(set_agencia - set_correo)
print(set_correo -set_agencia )

{'738381 - ENMANCIPACION', '739580 - ICA', '738360 - MOSHOQUEQUE', '737883 - COMAS', '738397 - AREQ CAYMA', '737896 - SAN JUAN DE LURIG', '739629 - ATE VITARTE', '739467 - HUACHO', '738224 - SAN JUAN DE MIRAFLORES', '738363 - CAJAMARCA', '735986 - JULIACA 2', '732243 - CAÑETE', '733825 - IQUITOS', '739483 - SAN MIGUEL', '738364 - TRUJ AMERICA', '734281 - CHICLAYO BALTA', '738371 - SAN MARTIN', '730109 - VENTANILLA', '737166 - MIRAFLORES', '739849 - LOS OLIVOS', '732000 - TUMBES', '734272 - CHIMBOTE', '734285 - PC HUARAZ', '738369 - PUENTE PIEDRA', '738382 - JESUS MARIA', '734280 - PC HUANCAYO', '734299 - CUSCO LA CULTURA', '739470 - HUARAL', '738252 - SANTA ANITA', '730879 - PAITA', '736568 - AREQ PAMPILLA', '738334 - PUCALLPA', '734264 - PISCO', '735996 - HUANUCO', '738391 - CHINCHA', '734265 - TRUJ CENTRO', '738013 - PC TACNA', '733824 - TARAPOTO', '734270 - SULLANA', '737490 - CASTILLA'}
set()


In [24]:
df_formato['agencia_tienda'] = (
    df_formato['agencia_atencion']
    .replace("CAÃ‘ETE", "CAÑETE")
)

In [ ]:
df_formato = df_formato[
    ~df_formato['agencia_tienda'].isin(['CASTILLA', 'AREQUIPA PAMPILLA', '734281 -  CHICLAYO BALTA', 'JESUS MARIA'])
].copy()

In [19]:
print(df_agencia["agencia_correo"].drop_duplicates().tolist())

['CAJAMARCA', 'CASTILLA', 'CHICLAYO BALTA', 'CHIMBOTE', 'MOSHOQUEQUE', None, 'HUARAZ', 'SULLANA', 'TRUJILLO AMERICA', 'TRUJILLO CENTRO', 'AREQUIPA CAYMA', 'AREQUIPA PAMPILLA', 'CAÑETE', 'CHINCHA', 'CUSCO LA CULTURA', 'HUACHO', 'HUANUCO', 'HUARAL', 'ICA', 'IQUITOS', 'JULIACA 2', 'HUANCAYO', 'TACNA', 'PISCO', 'PUCALLPA', 'TARAPOTO', 'ATE VITARTE', 'COMAS', 'EMANCIPACION', 'JESUS MARIA', 'LOS OLIVOS', 'MIRAFLORES', 'PUENTE PIEDRA', 'SAN JUAN DE LURIGANCHO', 'SAN JUAN DE MIRAFLORES', 'SAN MARTIN', 'SAN MIGUEL', 'SANTA ANITA', 'VENTANILLA', 'VILLA EL SALVADOR 2', 'VILLA MARIA 2', 'TUMBES']


In [ ]:
['CAJAMARCA', 'CASTILLA', 'CHICLAYO BALTA', 'CHIMBOTE', 'MOSHOQUEQUE', None, 'HUARAZ', 'SULLANA', 'TRUJILLO AMERICA', 'TRUJILLO CENTRO', 'AREQUIPA CAYMA', 'AREQUIPA PAMPILLA', 'CAÑETE', 'CHINCHA', 'CUSCO LA CULTURA', 'HUACHO', 'HUANUCO', 'HUARAL', 'ICA', 'IQUITOS', 'JULIACA 2', 'HUANCAYO', 'TACNA', 'PISCO', 'PUCALLPA', 'TARAPOTO', 'ATE VITARTE', 'COMAS', 'EMANCIPACION', 'JESUS MARIA', 'LOS OLIVOS', 'MIRAFLORES', 'PUENTE PIEDRA', 'SAN JUAN DE LURIGANCHO', 'SAN JUAN DE MIRAFLORES', 'SAN MARTIN', 'SAN MIGUEL', 'SANTA ANITA', 'VENTANILLA', 'VILLA EL SALVADOR 2', 'VILLA MARIA 2', 'TUMBES']MARIA 

In [27]:

equivalencias = {
    'SAN JUAN DE LURIG': 'SAN JUAN DE LURIGANCHO',
    'ENMANCIPACION': 'EMANCIPACION',
    'PC HUANCAYO': 'HUANCAYO',
    'PC TACNA': 'TACNA',
    'PC HUARAZ': 'HUARAZ',
    'TRUJ CENTRO': 'TRUJILLO CENTRO',
    'TRUJ AMERICA': 'TRUJILLO AMERICA',
    'AREQ CAYMA': 'AREQUIPA CAYMA',
    'AREQ PAMPILLA': 'AREQUIPA PAMPILLA'
}

df_formato['agencia_atencion'] = (
    df_formato['agencia_atencion']
    .replace(equivalencias)
)
# df_formato.drop_duplicates(subset=["dni"], inplace=True)

In [34]:
print(df_correo["agencia_atencion"].drop_duplicates().tolist())


['HUANUCO', 'HUANCAYO', 'SAN MIGUEL', 'CUSCO LA CULTURA', 'TACNA', 'TRUJILLO CENTRO', 'TARAPOTO', 'EMANCIPACION', 'CHIMBOTE', 'MIRAFLORES', 'TRUJILLO AMERICA', 'HUACHO', 'COMAS', 'CHICLAYO BALTA', 'SAN JUAN DE LURIGANCHO', 'CASTILLA', 'AREQUIPA PAMPILLA', 'SULLANA', 'VENTANILLA', 'MOSHOQUEQUE', 'JESUS MARIA', 'PUCALLPA', 'ATE VITARTE', 'LOS OLIVOS', 'VILLA MARIA 2', 'SANTA ANITA', 'CAJAMARCA', 'AREQUIPA CAYMA', 'PISCO', 'HUARAZ', 'SAN JUAN DE MIRAFLORES', 'CHINCHA', 'HUARAL', 'ICA', 'SAN MARTIN', 'JULIACA 2', 'VILLA EL SALVADOR 2', 'TUMBES', 'CAÑETE', 'PUENTE PIEDRA', nan, 'TE']


In [ ]:

df_correo[df_correo['dni_cliente']=='09704310'].head()

,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
2190,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,09704310,SALVADOR ALBERTO CHOQUE ALARCON,NaN,18000,930162239,MIRAFLORES,2026-07-15,13:30:00,MANUAL


In [83]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['CANAL']='CANAL'
filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','CANALVENTA']].copy()
df_target_desembolso['DNI'] = (
    df_target_desembolso['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)
df_fugas['DNI'] = (
    df_fugas['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)

df_desembolso=df_fugas.merge(
    df_target_desembolso,
    on=['DNI'],
    how='left'
)
df_desembolso = df_desembolso.fillna("OTROS")
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)


filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

# filename='desembolso.csv'
# ruta_archivo = os.path.join(ruta_alfin, filename)
# df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})

# Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)





C:\Users\DATA\AppData\Local\Temp\ipykernel_8792\2356303307.py:79: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


In [11]:
df_retiros.columns

Index(['dni_cliente', 'celular'], dtype='object')

In [57]:

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())
dni_desembolso = set(df_desembolso['dni_cliente'].dropna())


In [4]:
df_retiros[df_retiros['dni_cliente']=='41999928'].head()

,dni_cliente,celular


In [32]:
filename='alfin_ult_2.csv'
ruta_archivo = os.path.join(ruta_csv, filename)
df_qui = pd.read_csv(ruta_archivo,sep=';')

In [35]:
df_qui.rename(columns={'DNI':'dni_cliente'},inplace=True)

In [37]:
df_ref=df_qui[['dni_cliente','COLOR_FINAL']].copy()

In [46]:
df_formato['dni_cliente'] = (
    df_formato['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)

In [47]:
df_formato_1=df_formato.merge(df_ref,on='dni_cliente',how='inner')

In [48]:

df_formato_1.count()

dni_cliente            2849
nombre_cliente         2849
celular                2849
cod_agencia            2849
agencia_atencion       2849
fecha_visita           2849
monto_solicitado       2849
color_1                   0
supervisor             2849
canal_campo            2849
codigo_ejecutivo_id    2849
ejecutivo_target       2849
cdv_alfin_banco        2849
hora_visita            2849
telefono_cliente       2849
dni_vendedor           2849
agencia_tienda         2849
operador               2849
tipo_gestion           2849
COLOR_FINAL            2849
dtype: int64

In [22]:
filename='quitar_.xlsx'
filePath = os.path.join(ruta_csv, filename)
df_haber = pd.read_excel(filePath)


In [9]:
df_prospectos_envio.shape


(21149, 21)

In [59]:
df_seg=df_seg[
    ~df_seg['dni_cliente'].isin(dni_retiro)&
    ~df_seg['celular'].isin(cel_retiro)&
    ~df_seg['dni_cliente'].isin(dni_desembolso)
    ].copy()
df_seg.shape


(67, 21)

In [ ]:

ruta_archivo = os.path.join(ruta_csv, 'no_no_no.csv')
df_prospectos_envio.to_csv(ruta_archivo, sep=';')

In [24]:
df_haber.columns

Index(['vendor_lead_code', 'phone_number'], dtype='object')

In [23]:
df_haber.shape

(13888, 2)

In [82]:
server_sql = server_zeus
db_sql = "THOTH"
user_sql = user_zeus
pwd_sql = pwd_zeus

engine = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
query = f"""
	SELECT distinct Dni,Descripcion_ FROM THOTH.dbo.Tmp_LLamadas_Alfin 
    where Descripcion_ in(
    'TELEFONO FUERA DE SERVICIO / NO EXISTE',
       'EXPRESO RECIBIR MÚLTIPLES LLAMADAS',
       'SOLICITÓ NO SER CONTACTADO',
       'FUERA DE SERVICIO',
       'EXPRESO FUTURA DENUNCIA ANTE INDECOPI O REGULADOR',
       'EXPRESO QUE NO AUTORIZÓ USO DE DATOS PERSONALES'
    )
"""
df_tipis = pd.read_sql(query, engine)
set_tipi = set(
    df_tipis['Dni']
    .dropna()
    .drop_duplicates()
)



## aca

In [76]:
query = f"""
	SELECT dni_cliente,'enviado' as estado17 FROM Alice.prospectos_envio_alfin 
    where estado='procesado'
    and fecha_envio>='2026-08-01'
"""
df_prospectos_envio_alfin = pd.read_sql(query, engine_mysql)

query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where estado='enviado'
    and fecha_envio>='2026-08-01'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)
df_seg=df_prospectos_correos_alfin.merge(df_prospectos_envio_alfin,on='dni_cliente',how='inner')
df_seg.shape

(147548, 22)

In [4]:
df_estado = (
    df_prospectos_correos_alfin
    .groupby('estado')
    .size()
    .reset_index(name='cantidad')
)

df_estado

,estado,cantidad
0,ENVIADO,41502


In [77]:
df_seg["q_envio"] = (
    df_seg.groupby("dni_cliente")["dni_cliente"]
    .transform("count")
)

In [78]:
df_seg["cantidad_repeticiones"] = (
    df_seg
    .groupby("dni_cliente")["dni_cliente"]
    .transform("count")
)

df_seg["fecha_envio"] = pd.to_datetime(
    df_seg["fecha_envio"],
    errors="coerce"
)
df_seg = df_seg.sort_values(
    "fecha_envio",
    ascending=False
)
df_seg = (
    df_seg
    .drop_duplicates(
        subset="dni_cliente",
        keep="first"
    )
    .reset_index(drop=True)
)

In [79]:
import pandas as pd

# Convertir a fecha
df_seg["fecha_envio"] = pd.to_datetime(
    df_seg["fecha_envio"],
    errors="coerce"
)

# Cantidad de días desde fecha_envio hasta hoy
df_seg["q_dias"] = (
    pd.Timestamp.today().normalize()
    - df_seg["fecha_envio"].dt.normalize()
).dt.days

In [80]:
df_seg["fecha_envio"] = pd.to_datetime(
    df_seg["fecha_envio"],
    errors="coerce"
)

condicion = (
    (
        (df_seg["q_envio"] > 1)
        &
        (df_seg["fecha_envio"].dt.normalize().isin([
            "2026-08-20"
        ]))
    )
    |
    (
        (df_seg["q_envio"] == 1)
        &
        (df_seg["fecha_envio"].dt.normalize() >= "2026-08-17")
    )
)

df_seg = df_seg[condicion].copy()

C:\Users\DATA\AppData\Local\Temp\ipykernel_8792\816224503.py:10: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  (df_seg["fecha_envio"].dt.normalize().isin([


In [81]:
df_seg.shape

(2362, 25)

In [ ]:
df_seg

In [7]:
df_seg.shape

(2374, 22)

In [76]:
df.shape

(7133, 21)

In [85]:
df_seg=df_seg[
    ~df_seg['dni_cliente'].isin(dni_retiro)&
    ~df_seg['celular'].isin(cel_retiro)&
    ~df_seg['dni_cliente'].isin(dni_desembolso)&
    ~df_seg['dni_cliente'].isin(set_tipi)
    ].copy()
df_seg.shape


(2286, 25)

In [74]:

ruta_archivo = os.path.join(ruta_csv, 'sssssubir.csv')
df.to_csv(ruta_archivo, sep=';')

In [48]:
df_seg["fecha_envio"] = pd.to_datetime(
    df_seg["fecha_envio"],
    errors="coerce"
)
df_seg[df_seg['fecha_envio'].isin(['2026-08-15','2026-08-17'])].shape


C:\Users\DATA\AppData\Local\Temp\ipykernel_7668\1060216791.py:5: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df_seg[df_seg['fecha_envio'].isin(['2026-08-15','2026-08-17'])].shape


(0, 24)

In [49]:
df_seg.shape


(2626, 24)

In [ ]:
query = f"""
    where fecha_envio>='2026-08-01'
	SELECT * FROM Alice.prospectos_correos_alfin 
"""
df_prospectos_envio = pd.read_sql(query, engine_mysql)

df_prospectos_envio=df_prospectos_envio.drop_duplicates('dni_cliente')

ruta_archivo = os.path.join(ruta_csv, 'subir_correo.csv')
df_prospectos_envio.to_csv(ruta_archivo, sep=';')


In [ ]:
df_prospectos_envio

In [51]:
df_prospectos_envio=df_prospectos_envio.merge(df_seg[['dni_cliente']],on='dni_cliente',how='inner')
df_prospectos_envio = df_prospectos_envio.drop_duplicates(subset="dni_cliente")
df_prospectos_envio.shape


(2626, 16)

In [ ]:
df_formato_1=df_formato_1.rename(columns={'COLOR_FINAL':'color'})

In [67]:
df_correo = df_correo.drop_duplicates(subset=['dni_cliente'])

df_formulario = df_formulario.drop_duplicates(subset=['dni_cliente'])

In [70]:
print(df_correo.shape)
print(df_formulario.shape)

(67, 14)
(67, 9)


In [94]:
df_correo=df_seg[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita','color']] .copy()
df_correo['tipo_carga']='MANUAL'

df_formulario=df_seg_formulario[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()

display(df_correo.head(2))
display(df_formulario.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,color,tipo_carga
2531,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,45055296,MARCOS EDGARDO ARQUINIGO ZAPANA,2200.0,992904622,VILLA MARIA 2,2026-08-22,18:15:00,AMARILLO OSCURO,MANUAL
2532,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,44441838,PAULA LORENZA MALDONADO BOCANEGRA,10000.0,940606019,VILLA EL SALVADOR 2,2026-08-22,17:00:00,AMARILLO OSCURO,MANUAL


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,80644018,LUIS ALBERTO ALCANTARA CAPU+æAY\t,940755112,734281 - CHICLAYO BALTA,2026-08-24,10000,Derivacion
1,BOT,TARGET,80234657,MARIA TEMPORA TIMANA LACHIRA,956619138,732243 - CAÃ‘ETE,2026-08-22,11000,Derivacion


In [86]:
import pandas as pd
import numpy as np

fechas = pd.to_datetime([
    "2026-08-22",
    "2026-08-24",
])

df_seg["fecha_visita"] = np.random.choice(
    fechas,
    size=len(df_seg)
)

In [87]:
# Horas posibles: 09 a 18
horas = np.random.randint(9, 19, size=len(df_seg))

# Minutos posibles
minutos = np.random.choice([0, 15, 30, 45], size=len(df_seg))

# Crear la columna
df_seg["hora_visita"] = [
    f"{h:02d}:{m:02d}:00"
    for h, m in zip(horas, minutos)
]

In [88]:
df=df_seg[["fecha_visita",'dni_cliente']].copy()

In [89]:
query = f"""
	SELECT * FROM Alice.prospectos_envio_alfin 
    where fecha_envio>='2026-08-01'
"""
df_seg_formulario = pd.read_sql(query, engine_mysql)

In [90]:
df_seg_formulario = df_seg_formulario.drop(
    columns=["fecha_visita"]
)

In [91]:
df_seg_formulario=df_seg_formulario.merge(df,on='dni_cliente',how='inner')
df_seg_formulario=df_seg_formulario.drop_duplicates(subset=['dni_cliente'])

In [92]:
df_seg_formulario.head(2)

,id,hash_duplicado,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,monto_solicitado,tipo_gestion,estado,codigo_http_ms,respuesta_ms,fecha_creacion,fecha_envio,fecha_visita
0,222560,None,BOT,TARGET,80644018,LUIS ALBERTO ALCANTARA CAPU+æAY\t,940755112,734281 - CHICLAYO BALTA,10000,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-08-02 06:00:00,2026-08-02 12:02:02,2026-08-24
1,222577,None,BOT,TARGET,80234657,MARIA TEMPORA TIMANA LACHIRA,956619138,732243 - CAÃ‘ETE,11000,Derivacion,PROCESADO,201,Registrado en Forms Alfin.,2026-08-01 07:52:43,2026-08-01 20:18:20,2026-08-22


In [93]:
df_seg_formulario.shape

(2286, 16)

In [28]:
df_seg.shape

(4211, 25)

In [95]:

df_correo.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_formulario.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

2286

In [22]:
df_correo_pendiente=df_correo.copy()
df_formulario_pendiente=df_formulario.copy()

In [44]:
df_formulario_pendiente[df_formulario_pendiente['dni_cliente']=='80248715'].head()

,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion


In [45]:
df_formulario[df_formulario['dni_cliente']=='80248715'].head()


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
15,BOT,TARGET,80248715,YOLANDA MARIA OLIVERO AGUILAR,991554388,737166 - MIRAFLORES,2026-08-02,4700,Derivacion


In [ ]:
df_prospectos_envio=df_prospectos_envio.drop_duplicates('dni_cliente')


In [75]:

ruta_archivo = os.path.join(ruta_csv, 'subir_vici.csv')
df.to_csv(ruta_archivo, sep=';')


In [9]:
df_seg.shape

(2374, 22)

In [46]:
df_formulario_actualizado = pd.read_csv(ruta_archivo,sep=';')

In [ ]:
df_formulario_actualizado['fecha_visita']=df_formulario_actualizado['fecha_visita']

In [51]:
df_formulario_actualizado["fecha_visita"] = pd.to_datetime(
    df_formulario_actualizado["fecha_visita"],
    errors="coerce"
)

In [ ]:
df_formulario_actualizado

In [52]:
df_formulario_actualizado.head()

,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,BOT,TARGET,48124718,CHOQUE CURO NOEMY LUZMERY,937616610,737870 - VILLA MARIA 2,2026-06-08,20000,Derivacion
1,BOT,TARGET,41494656,GUTIERREZ HEREDIA CARLOS ALBERTO,994746090,737870 - VILLA MARIA 2,2026-03-08,19000,Derivacion
2,BOT,TARGET,41298318,ROXANA CELIA ESPINOZA MALLCO,970551073,737870 - VILLA MARIA 2,2026-05-08,4900,Derivacion
3,BOT,TARGET,40865470,PEREZ ALVA EDWUAR RICARDO,931126400,737870 - VILLA MARIA 2,2026-03-08,9000,Derivacion
4,BOT,TARGET,10098109,FLOR DE MARIA PAZ,913923143,737870 - VILLA MARIA 2,2026-03-08,20000,Derivacion


In [48]:
df_formulario_actualizado['dni_cliente'] = (
    df_formulario_actualizado['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)      # deja solo números
    .replace('', pd.NA)                      # vacío -> NA
    .mask(lambda s: s.str.len() > 8, pd.NA)  # >8 dígitos -> NA
    .str.zfill(8)                            # <8 dígitos -> completa con ceros
)

In [72]:
query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where fecha_registro>='2026-08-21'
"""
df = pd.read_sql(query, engine_mysql)
df.shape

(5071, 21)

In [73]:
df=df.drop_duplicates(['dni_cliente'])


(5053, 21)

In [57]:
df_seg_formulario =df_prospectos_envio_alfin.merge(df_seg[['dni_cliente']],on='dni_cliente',how='inner')

In [58]:
df_seg_formulario=df_seg_formulario.drop_duplicates('dni_cliente')


In [59]:
df_seg_formulario.columns

Index(['id', 'hash_duplicado', 'dni_vendedor', 'operador', 'dni_cliente',
       'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita',
       'monto_solicitado', 'tipo_gestion', 'estado', 'codigo_http_ms',
       'respuesta_ms', 'fecha_creacion', 'fecha_envio'],
      dtype='object')

In [ ]:
df_correo = df_correo.drop_duplicates(subset=['dni_cliente'])

df_formulario = df_formulario.drop_duplicates(subset=['dni_cliente'])

In [69]:
dni_s=set(df['dni_cliente'].unique())

In [99]:
dni_s=set(df_no_Cargar['DNI'].unique())


In [100]:
df=df[~df['dni_cliente'].isin(dni_s)].copy()


In [ ]:
# dni_s=set(df['dni_cliente'].unique())
df_correo=df_correo[~df_correo['dni_cliente'].isin(dni_s)].copy()
df_formulario=df_formulario[~df_formulario['dni_cliente'].isin(dni_s)].copy()

In [101]:
df.shape

(6599, 21)

In [90]:
filename='NO CARGAR.xlsx'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_no_Cargar = pd.read_excel(ruta_archivo)

In [94]:
df_no_Cargar['DNI'] = (
    df_no_Cargar['DNI']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.replace(r'\D', '', regex=True)
    .str.zfill(8)                            # <8 dígitos -> completa con ceros
)

In [95]:
df_no_Cargar.head(2)

,DNI
0,09891347
1,43173825


In [ ]:
df_no_Cargar['DNI'] = (
    df_no_Cargar['DNI']
    .astype(str)
    
    .replace('', pd.NA)                      # vacío -> NA
    .str.zfill(8)                            # <8 dígitos -> completa con ceros

)

In [80]:
df_no_Cargar['DNI'] = (
    df_no_Cargar['DNI']
    .astype(str)
    .str.replace(r'\D', '', regex=True)      # deja solo números
    .replace('', pd.NA)                      # vacío -> NA
    .mask(lambda s: s.str.len() > 8, pd.NA)  # >8 dígitos -> NA
    .str.zfill(8)                            # <8 dígitos -> completa con ceros
)